# Willow v6 RAG Integration Example

This notebook demonstrates how to use the RAG (Retrieval-Augmented Generation) functionality in Willow v6.

## Overview

Willow v6 provides multiple ways to integrate RAG functionality:
- **Simple RAG Pipeline**: Basic document storage and retrieval
- **LangChain Integration**: Full-featured RAG with advanced capabilities
- **Async Operations**: High-performance async/await support
- **Graceful Degradation**: Works even when dependencies are missing


## Setup and Dependencies

First, let's install the required dependencies and set up the environment.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install faiss-cpu sentence-transformers
# !pip install langchain langchain-community  # For full LangChain integration
# !pip install datasets ragas  # For evaluation metrics

import sys
import os
import asyncio
from pathlib import Path
import tempfile
import json

# Add Willow src to path
willow_root = Path.cwd().parent if 'examples' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(willow_root / 'src'))

print(f"Willow root: {willow_root}")
print(f"Python path: {sys.path[:2]}")

## Check RAG Availability

Let's check what RAG functionality is available in your environment.

In [ ]:
# Check RAG availability
try:
    from main import RAG_AVAILABLE
    print(f"🔍 RAG Available in main.py: {RAG_AVAILABLE}")
except ImportError as e:
    print(f"❌ Could not import from main.py: {e}")
    RAG_AVAILABLE = False

# Check Vector DB availability
try:
    from utils.vector_db import VECTOR_DB_AVAILABLE
    print(f"🔍 Vector DB Available: {VECTOR_DB_AVAILABLE}")
except ImportError as e:
    print(f"❌ Could not import vector_db: {e}")
    VECTOR_DB_AVAILABLE = False

# Check simple RAG pipeline
try:
    from rag_pipeline import create_rag_pipeline, RAGPipeline
    print("✅ Simple RAG Pipeline available")
    SIMPLE_RAG_AVAILABLE = True
except ImportError as e:
    print(f"❌ Simple RAG Pipeline not available: {e}")
    SIMPLE_RAG_AVAILABLE = False

print("\n📊 RAG Capabilities Summary:")
print(f"   Simple RAG: {'✅' if SIMPLE_RAG_AVAILABLE else '❌'}")
print(f"   Vector DB: {'✅' if VECTOR_DB_AVAILABLE else '❌'}")
print(f"   Full Integration: {'✅' if RAG_AVAILABLE else '❌'}")

## Example 1: Simple RAG Pipeline

Let's start with the simplest RAG usage - creating a pipeline and adding some documents.

In [ ]:
if SIMPLE_RAG_AVAILABLE:
    # Create a temporary directory for our RAG database
    temp_dir = tempfile.mkdtemp()
    db_path = Path(temp_dir) / "rag_demo_db"
    
    print(f"📁 Using temporary database: {db_path}")
    
    try:
        # Create RAG pipeline
        rag = create_rag_pipeline(db_path=db_path)
        print("✅ RAG pipeline created successfully")
        
        # Sample documents about Willow
        documents = [
            "Willow v6 is an advanced AI automation framework with RAG capabilities.",
            "The system supports multiple LLM providers including OpenAI, Anthropic, and local models.",
            "RAG (Retrieval-Augmented Generation) enhances responses with relevant context from documents.",
            "Willow includes security features like input validation and path sanitization.",
            "The framework supports async operations for high-performance applications.",
            "Vector databases enable semantic search and similarity matching in Willow."
        ]
        
        # Metadata for documents
        metadata = [
            {"topic": "overview", "category": "introduction", "id": 1},
            {"topic": "llm_providers", "category": "features", "id": 2},
            {"topic": "rag", "category": "features", "id": 3},
            {"topic": "security", "category": "features", "id": 4},
            {"topic": "performance", "category": "features", "id": 5},
            {"topic": "vector_db", "category": "technology", "id": 6}
        ]
        
        # Add documents to the pipeline
        rag.add_documents(documents, metadata)
        print(f"✅ Added {len(documents)} documents to RAG pipeline")
        
        # Get pipeline statistics
        stats = rag.get_stats()
        print(f"📊 Pipeline stats: {stats}")
        
    except Exception as e:
        print(f"❌ Error setting up RAG pipeline: {e}")
        rag = None
else:
    print("⚠️ Simple RAG not available - skipping this example")
    rag = None

## Example 2: Querying Documents

Now let's query our RAG pipeline to retrieve relevant documents.

In [ ]:
if rag is not None:
    # Test queries
    test_queries = [
        "What is Willow?",
        "How does RAG work?",
        "What security features are available?",
        "Tell me about performance"
    ]
    
    for i, query in enumerate(test_queries, 1):
        print(f"\n🔍 Query {i}: '{query}'")
        print("-" * 50)
        
        try:
            # Basic query
            results = rag.query(query, top_k=2)
            
            print(f"📄 Found {len(results)} relevant documents:")
            for j, result in enumerate(results, 1):
                content = result['content'][:100] + "..." if len(result['content']) > 100 else result['content']
                score = result['relevance_score']
                metadata = result['metadata']
                
                print(f"   {j}. Score: {score:.3f}")
                print(f"      Content: {content}")
                print(f"      Topic: {metadata.get('topic', 'N/A')}")
                
        except Exception as e:
            print(f"❌ Query failed: {e}")
else:
    print("⚠️ No RAG pipeline available for querying")

## Example 3: Context Formatting for LLM

RAG pipelines can format retrieved context for use with LLMs.

In [ ]:
if rag is not None:
    query = "What are the main features of Willow v6?"
    
    try:
        # Get formatted context
        context_result = rag.query_with_context(query, top_k=3)
        
        print(f"🎯 Query: '{query}'")
        print(f"📊 Document count: {context_result['document_count']}")
        print("\n📋 Formatted Context for LLM:")
        print("=" * 80)
        print(context_result['context'])
        print("=" * 80)
        
        # Show individual documents
        print("\n📄 Retrieved Documents:")
        for i, doc in enumerate(context_result['documents'], 1):
            print(f"   {i}. {doc['content']}")
            print(f"      Metadata: {doc['metadata']}")
            
    except Exception as e:
        print(f"❌ Context formatting failed: {e}")
else:
    print("⚠️ No RAG pipeline available for context formatting")

## Example 4: Async RAG Operations

Willow supports async operations for better performance.

In [ ]:
import time

async def demo_async_rag():
    """Demonstrate async RAG operations."""
    if rag is None:
        print("⚠️ No RAG pipeline available for async demo")
        return
    
    print("🚀 Testing Async RAG Operations")
    print("=" * 40)
    
    # Batch queries
    queries = [
        "What is RAG?",
        "Security features",
        "Performance optimization",
        "LLM providers"
    ]
    
    # Sequential queries
    start_time = time.perf_counter()
    sequential_results = []
    
    for query in queries:
        try:
            result = await rag.query_async(query, top_k=1)
            sequential_results.append(result)
        except Exception as e:
            print(f"Sequential query failed: {e}")
            # Fallback to sync version
            result = rag.query(query, top_k=1)
            sequential_results.append(result)
    
    sequential_time = time.perf_counter() - start_time
    
    # Concurrent queries
    start_time = time.perf_counter()
    
    try:
        # Try async concurrent queries
        tasks = [rag.query_async(query, top_k=1) for query in queries]
        concurrent_results = await asyncio.gather(*tasks)
    except Exception as e:
        print(f"Concurrent async queries failed: {e}")
        # Fallback to sequential sync queries
        concurrent_results = [rag.query(query, top_k=1) for query in queries]
    
    concurrent_time = time.perf_counter() - start_time
    
    # Results
    print(f"⏱️ Sequential time: {sequential_time:.3f}s")
    print(f"⏱️ Concurrent time: {concurrent_time:.3f}s")
    
    if concurrent_time < sequential_time:
        speedup = sequential_time / concurrent_time
        print(f"🚀 Speedup: {speedup:.2f}x faster with concurrent queries")
    else:
        print("📊 No significant speedup (may be due to small dataset or sync fallback)")
    
    return sequential_results, concurrent_results

# Run async demo
if SIMPLE_RAG_AVAILABLE:
    try:
        # For Jupyter notebook, we need to handle the event loop carefully
        if asyncio.get_event_loop().is_running():
            # Already in an async context, just await
            results = await demo_async_rag()
        else:
            # Create new event loop
            results = asyncio.run(demo_async_rag())
    except RuntimeError:
        # Fallback for Jupyter
        print("🔄 Running in compatibility mode...")
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            results = loop.run_until_complete(demo_async_rag())
        finally:
            loop.close()
else:
    print("⚠️ Async demo skipped - RAG not available")

## Example 5: Integration with Main Willow Functions

Let's test the full integration with Willow's main functions.

In [ ]:
async def demo_main_integration():
    """Demonstrate integration with main Willow functions."""
    print("🔗 Testing Main Willow Integration")
    print("=" * 40)
    
    if not RAG_AVAILABLE:
        print("⚠️ Full RAG integration not available")
        print("📦 To enable: pip install langchain langchain-community")
        return
    
    try:
        from main import async_query_rag_simple
        
        # Test with our existing documents
        test_documents = [
            "Willow v6 is an AI framework with advanced RAG capabilities.",
            "The system supports both local and cloud-based LLM providers.",
            "Security is a core feature with input validation and sandboxing."
        ]
        
        query = "What are the key features of Willow?"
        
        print(f"🎯 Query: '{query}'")
        print(f"📚 Using {len(test_documents)} documents")
        
        # This would normally generate an LLM response
        # For demo purposes, we'll show the process
        response = await async_query_rag_simple(
            query, 
            documents=test_documents,
            top_k=2
        )
        
        print(f"🤖 Response: {response[:200]}...")
        print("✅ Main integration working")
        
    except Exception as e:
        print(f"❌ Main integration failed: {e}")
        print("💡 This might be expected if LLM providers are not configured")

# Run main integration demo
if RAG_AVAILABLE:
    try:
        await demo_main_integration()
    except Exception as e:
        print(f"❌ Demo failed: {e}")
else:
    print("⚠️ Main integration demo skipped - dependencies not available")

## Example 6: Performance Monitoring

Monitor RAG performance and get detailed metrics.

In [ ]:
if rag is not None:
    # Performance monitoring
    print("📊 RAG Performance Monitoring")
    print("=" * 40)
    
    # Get detailed statistics
    stats = rag.get_stats()
    print("📈 Current Statistics:")
    for key, value in stats.items():
        print(f"   {key}: {value}")
    
    # Health check
    try:
        health = rag.health_check()
        print("\n🏥 Health Check:")
        for key, value in health.items():
            status = "✅" if value else "❌"
            print(f"   {status} {key}: {value}")
    except Exception as e:
        print(f"❌ Health check failed: {e}")
    
    # Performance test
    print("\n⚡ Performance Test:")
    
    queries = ["test query"] * 10  # Repeat same query to test caching
    
    start_time = time.perf_counter()
    for i, query in enumerate(queries):
        results = rag.query(query, top_k=1)
        if i == 0:
            print(f"   First query: {len(results)} results")
    
    total_time = time.perf_counter() - start_time
    avg_time = total_time / len(queries)
    
    print(f"   Total time for {len(queries)} queries: {total_time:.3f}s")
    print(f"   Average time per query: {avg_time:.3f}s")
    
    if avg_time < 0.1:
        print("   🚀 Excellent performance (likely cached)")
    elif avg_time < 0.5:
        print("   ✅ Good performance")
    else:
        print("   ⚠️ Consider optimizing for better performance")
else:
    print("⚠️ Performance monitoring skipped - no RAG pipeline available")

## Example 7: Error Handling and Graceful Degradation

Demonstrate how Willow handles errors and degrades gracefully.

In [ ]:
print("🛡️ Error Handling and Graceful Degradation")
print("=" * 50)

# Test with missing dependencies
print("🔍 Testing graceful degradation:")

try:
    # This should work even with missing dependencies
    from rag_pipeline import RAGError
    print("   ✅ RAG error classes available")
except ImportError:
    print("   ✅ Graceful import degradation working")

# Test configuration loading
try:
    import json
    with open('../config_rag.json', 'r') as f:
        config = json.load(f)
    print("   ✅ RAG configuration loaded")
    print(f"   📋 Config keys: {list(config.keys())}")
except FileNotFoundError:
    print("   ⚠️ RAG config file not found (this is OK for demo)")
except Exception as e:
    print(f"   ❌ Config loading failed: {e}")

# Test retry logic
try:
    from utils.retry_logic import RAGRetryConfig, with_retry
    retry_config = RAGRetryConfig(max_retries=2)
    print("   ✅ Retry logic available")
    print(f"   ⚙️ Max retries: {retry_config.max_retries}")
except ImportError:
    print("   ⚠️ Retry logic not available")

# Summary
print("\n📋 Summary:")
print(f"   Simple RAG: {'✅' if SIMPLE_RAG_AVAILABLE else '❌'}")
print(f"   Vector DB: {'✅' if VECTOR_DB_AVAILABLE else '❌'}")
print(f"   Full Integration: {'✅' if RAG_AVAILABLE else '❌'}")

if not all([SIMPLE_RAG_AVAILABLE, VECTOR_DB_AVAILABLE, RAG_AVAILABLE]):
    print("\n💡 To enable full functionality:")
    if not VECTOR_DB_AVAILABLE:
        print("   pip install faiss-cpu sentence-transformers")
    if not RAG_AVAILABLE:
        print("   pip install langchain langchain-community")
    print("   pip install datasets ragas  # For evaluation")
else:
    print("\n🎉 All RAG functionality is available!")

## Cleanup

Clean up temporary files and resources.

In [ ]:
# Cleanup
import shutil

if 'temp_dir' in locals():
    try:
        shutil.rmtree(temp_dir)
        print(f"🧹 Cleaned up temporary directory: {temp_dir}")
    except Exception as e:
        print(f"⚠️ Cleanup warning: {e}")

print("\n✅ Notebook demonstration completed!")
print("\n📚 Next Steps:")
print("   1. Install full dependencies for complete functionality")
print("   2. Create your own documents and test queries")
print("   3. Integrate RAG into your own applications")
print("   4. Explore the configuration options in config_rag.json")
print("   5. Check out the retry logic and circuit breaker patterns")